# Precificação de Imóveis no Brasil: Análise Exploratória e Machine Learning

Este projeto aborda o ciclo completo de análise e modelagem preditiva no mercado imobiliário brasileiro:
1. **Análise Exploratória (EDA):** padrões de preços por metro quadrado, correlações e contrastes regionais.
2. **Engenharia de Recursos e Pré-processamento:** tratamento de variáveis, One-Hot Encoding e separação de treino/teste.
3. **Modelagem Preditiva:** regressão para estimar o valor de aluguel (`valor_aluguel`) comparando um modelo linear (baseline) com Random Forest.
4. **Diagnóstico e Interpretabilidade:** métricas de erro (MAE, RMSE, R²) e identificação dos atributos que mais agregam valor aos imóveis.

### 1. Importação de Bibliotecas

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

### 2. Leitura e Inspeção Geral dos Dados

In [ ]:
caminho_dados = os.path.join('dados', 'imoveis_brasil.csv')
df = pd.read_csv(caminho_dados)

print('Total de registros:', len(df))
print('Colunas:', list(df.columns))
df.head()

In [ ]:
# Resumo estatístico das variáveis contínuas e financeiras
cols_resumo = ['area_m2', 'quartos', 'banheiros', 'vagas_garagem', 'valor_condominio', 'valor_iptu', 'valor_aluguel', 'valor_venda']
df[cols_resumo].describe().round(2)

### 3. Análise Exploratória (EDA)

In [ ]:
# Preço médio do aluguel por m² por cidade
df['preco_m2_aluguel'] = df['valor_aluguel'] / df['area_m2']
media_cidades = df.groupby('cidade')['preco_m2_aluguel'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=media_cidades.values, y=media_cidades.index, palette='Blues_r', ax=ax)
ax.set_title('Preço Médio do Aluguel por m² por Cidade (R$ / m²)', fontsize=14, pad=12)
ax.set_xlabel('R$ / m²')
ax.set_ylabel('')
for i, val in enumerate(media_cidades.values):
    ax.text(val + 0.6, i, f'R$ {val:.2f}', va='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Relação entre Área Útil e Aluguel Mensal por Tipo de Imóvel
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=df.sample(2500, random_state=42),
    x='area_m2',
    y='valor_aluguel',
    hue='tipo_imovel',
    alpha=0.6,
    palette='tab10',
    ax=ax
)
ax.set_title('Área Útil vs. Valor do Aluguel por Categoria', fontsize=14, pad=12)
ax.set_xlabel('Área Útil (m²)')
ax.set_ylabel('Aluguel Mensal (R$)')
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de Correlação entre Variáveis Numéricas
corr_cols = ['area_m2', 'quartos', 'banheiros', 'vagas_garagem', 'valor_condominio', 'valor_iptu', 'valor_venda', 'valor_aluguel']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', cbar=True, square=True)
plt.title('Matriz de Correlação de Pearson', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

### 4. Preparação de Dados e Divisão Treino/Teste

In [ ]:
# Seleção das variáveis preditoras e do alvo
target = 'valor_aluguel'
features_num = ['area_m2', 'quartos', 'banheiros', 'vagas_garagem', 'andar', 'aceita_animais', 'mobiliado', 'tem_piscina', 'tem_academia', 'valor_condominio', 'valor_iptu']
features_cat = ['cidade', 'tipo_imovel']

# One-Hot Encoding nas variáveis categóricas
df_ml = pd.get_dummies(df[features_num + features_cat + [target]], columns=features_cat, drop_first=True)

X = df_ml.drop(columns=[target])
y = df_ml[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print('Registros de Treino:', len(X_train))
print('Registros de Teste: ', len(X_test))
print('Total de Features:  ', X_train.shape[1])

### 5. Treinamento e Comparativo de Modelos

In [ ]:
# 1. Regressão Linear Simples / Múltipla (Baseline)
modelo_lr = LinearRegression()
modelo_lr.fit(X_train, y_train)
y_pred_lr = modelo_lr.predict(X_test)

# 2. Random Forest Regressor
modelo_rf = RandomForestRegressor(n_estimators=100, max_depth=16, random_state=42, n_jobs=-1)
modelo_rf.fit(X_train, y_train)
y_pred_rf = modelo_rf.predict(X_test)

def calcular_metricas(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

mae_lr, rmse_lr, r2_lr = calcular_metricas(y_test, y_pred_lr)
mae_rf, rmse_rf, r2_rf = calcular_metricas(y_test, y_pred_rf)

resultados = pd.DataFrame({
    'Modelo': ['Regressão Linear (Baseline)', 'Random Forest Regressor'],
    'MAE (R$)': [mae_lr, mae_rf],
    'RMSE (R$)': [rmse_lr, rmse_rf],
    'R² Score': [r2_lr, r2_rf]
}).round(3)

resultados

### 6. Diagnóstico e Importância dos Atributos

In [ ]:
# Gráfico de Dispersão: Valores Reais vs. Previstos no Conjunto de Teste
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, y_pred_rf, alpha=0.3, color='#1f77b4', edgecolors='none')
lims = [0, max(y_test.max(), y_pred_rf.max()) * 1.05]
ax.plot(lims, lims, color='red', linestyle='--', linewidth=2, label='Linha Ideal (y = x)')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel('Valor Real do Aluguel (R$)')
ax.set_ylabel('Valor Previsto pelo Modelo (R$)')
ax.set_title('Random Forest: Aluguel Real vs. Aluguel Previsto', fontsize=14, pad=12)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Importância das Variáveis no Random Forest
importancias = pd.Series(modelo_rf.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(12)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=importancias.values, y=importancias.index, palette='viridis', ax=ax)
ax.set_title('Top 12 Variáveis com Maior Impacto no Preço do Aluguel', fontsize=14, pad=12)
ax.set_xlabel('Importância Relativa (Gini)')
plt.tight_layout()
plt.show()

### 7. Simulação Prática de Previsão
Testando o modelo treinado com os atributos de um novo imóvel hipotético no mercado.

In [ ]:
# Exemplo: Apartamento de 85m² em São Paulo, mobiliado, 2 quartos, 2 banheiros e 1 vaga
novo_imovel = pd.DataFrame(columns=X_train.columns, index=[0]).fillna(0)

novo_imovel['area_m2'] = 85
novo_imovel['quartos'] = 2
novo_imovel['banheiros'] = 2
novo_imovel['vagas_garagem'] = 1
novo_imovel['andar'] = 8
novo_imovel['aceita_animais'] = 1
novo_imovel['mobiliado'] = 1
novo_imovel['tem_piscina'] = 1
novo_imovel['tem_academia'] = 1
novo_imovel['valor_condominio'] = 850
novo_imovel['valor_iptu'] = 180
if 'cidade_São Paulo' in novo_imovel.columns:
    novo_imovel['cidade_São Paulo'] = 1

valor_estimado = modelo_rf.predict(novo_imovel)[0]
print(f'Aluguel estimado para o imóvel configurado: R$ {valor_estimado:,.2f}')